# Kimi K2.6 Agentic Workflow

**How to run:**
1. Add `TOGETHER_API_KEY` and `TAVILY_API_KEY` to your Kaggle secrets and enable notebook access.
2. Upload your `your_chats.json` to the notebook working directory (or a Kaggle dataset).
3. Run cells top to bottom.

In [ ]:
# Phase 1 — Install dependencies
%pip install -q requests tenacity pydantic

In [ ]:
# Phase 1 — Load secrets
from kaggle_secrets import UserSecretsClient

_secrets = UserSecretsClient()
TOGETHER_API_KEY = _secrets.get_secret("TOGETHER_API_KEY")
TAVILY_API_KEY   = _secrets.get_secret("TAVILY_API_KEY")

KIMI_BASE_URL = "https://api.together.ai/v1/chat/completions"
KIMI_MODEL    = "moonshotai/Kimi-K2.6"
TAVILY_URL    = "https://api.tavily.com/search"

In [ ]:
# Phase 2 — kimi_call
import requests
import tenacity

@tenacity.retry(
    wait=tenacity.wait_exponential(multiplier=1, min=2, max=8),
    stop=tenacity.stop_after_attempt(3),
    retry=tenacity.retry_if_exception_type(requests.exceptions.RequestException),
    reraise=True,
)
def kimi_call(messages, max_tokens=512):
    response = requests.post(
        KIMI_BASE_URL,
        headers={
            "Authorization": f"Bearer {TOGETHER_API_KEY}",
            "Content-Type": "application/json",
        },
        json={
            "model": KIMI_MODEL,
            "messages": messages,
            "temperature": 0.3,
            "max_tokens": max_tokens,
        },
        timeout=30,
    )
    response.raise_for_status()
    return response.json()["choices"][0]["message"]["content"]

In [ ]:
# Phase 2 — web_search
_search_cache = {}

@tenacity.retry(
    wait=tenacity.wait_exponential(multiplier=1, min=2, max=8),
    stop=tenacity.stop_after_attempt(3),
    retry=tenacity.retry_if_exception_type(requests.exceptions.RequestException),
    reraise=True,
)
def web_search(query, max_results=5):
    if query in _search_cache:
        return _search_cache[query]
    response = requests.post(
        TAVILY_URL,
        json={
            "api_key": TAVILY_API_KEY,
            "query": query,
            "max_results": max_results,
            "search_depth": "basic",
            "include_answer": False,
            "after_date": "2022-01-01",
        },
        timeout=10,
    )
    response.raise_for_status()
    results = response.json()
    _search_cache[query] = results
    return results

In [ ]:
# Phase 2 — count_tokens
def count_tokens(text):
    return int(len(text.split()) * 1.3)

In [ ]:
# Phase 2 — Smoke tests
# kimi_call
reply = kimi_call([{"role": "user", "content": "Reply with exactly: OK"}], max_tokens=10)
print("kimi_call :", reply)

# web_search
results = web_search("latest AI news", max_results=1)
print("web_search keys:", list(results.keys()))
print("first result  :", results["results"][0]["title"])

# count_tokens
print("count_tokens  :", count_tokens("hello world this is a test sentence"))